### Environment Setup & Imports

In [2]:
import os
import warnings
from dotenv import load_dotenv

# Suppress non-critical deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# LangChain & RAG dependencies
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv()

# Verify API Key
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("❌ GROQ_API_KEY not found in .env file. Please add your key.")
else:
    print("✅ Groq API Key loaded successfully.")

✅ Groq API Key loaded successfully.


### Document Ingestion

In [4]:
DOCS_DIR = "document_corpus"  # Directory containing your PDF files

print(f"Loading PDF files from '{DOCS_DIR}'...")
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

print(f"✅ Total document pages loaded: {len(documents)}")

if documents:
    print("\nSample Metadata from First Page:")
    print(documents[0].metadata)

Loading PDF files from 'document_corpus'...
✅ Total document pages loaded: 31

Sample Metadata from First Page:
{'producer': 'MiKTeX pdfTeX-1.40.25', 'creator': 'TeX', 'creationdate': '2026-10-06T16:23:29+05:30', 'moddate': '2026-10-06T16:23:29+05:30', 'trapped': '/False', 'ptex.fullbanner': 'This is MiKTeX-pdfTeX 4.18.0 (1.40.25)', 'source': 'document_corpus\\DOC_01_Return_and_Refund_Policy.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


### Text Chunking & Splitting

In [5]:
# Split long document pages into smaller overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)
print(f"✅ Generated {len(docs)} text chunks across all documents.")

print("\n--- Preview of First Chunk ---")
print(docs[0].page_content[:300] + "...")

✅ Generated 63 text chunks across all documents.

--- Preview of First Chunk ---
Document ID: DOC-01-RRP-2026 OmniTech Retail Solutions — Standard Operating Policy
utfcode
utf8.sty 3.10 UTF-8 input encoding 13.06.2000
scanner for code UTF-8 installed.
OmniTech Retail Solutions
Global Return, Refund, and Exchange Policy
Version 4.2 — Effective Date: January 1, 2026 — Document Ref...


### Embedding Generation & ChromaDB Persistence

In [ ]:
CHROMA_DIR = "data/chroma_db"

print("Initializing local HuggingFace embeddings ('all-MiniLM-L6-v2')...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Initializing local HuggingFace embeddings ('all-MiniLM-L6-v2')...


e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

e:\5th semester lecture notes\supervisor_agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings and saving into persistent ChromaDB...


NameError: name 'docs' is not defined

In [6]:
print("Generating embeddings and saving into persistent ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)

print(f"✅ Vector database successfully saved to '{CHROMA_DIR}'.")

Generating embeddings and saving into persistent ChromaDB...
✅ Vector database successfully saved to 'data/chroma_db'.


### Initialize Groq LLM & Construct LCEL RAG Chain (With Citations)

In [9]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize Groq LLM
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    temperature=0.0
)

# 2. Define System Prompt
prompt_template = """You are an assistant for answering questions based on official company documentation.
Use the provided context to answer the question accurately. If you do not know the answer based on the context, say so.

<context>
{context}
</context>

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Helper function to merge retrieved text chunks into context
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# 3. Configure Vector Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4. Construct LCEL Chain returning both generated answer AND source documents
rag_chain_with_sources = RunnableParallel(
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
).assign(
    answer=(
        RunnablePassthrough.assign(
            context=lambda x: format_docs(x["context"])
        )
        | prompt
        | llm
        | StrOutputParser()
    )
)

print("✅ LCEL RAG Pipeline with PDF source citations initialized!")

✅ LCEL RAG Pipeline with PDF source citations initialized!


### Run Queries & Display Answer with PDF Citations

In [10]:
query = "What is the return window and restocking fee for opened laptops?"
print(f"Query: {query}\n")

# Execute query against RAG pipeline
response = rag_chain_with_sources.invoke(query)

# Display Generated Response
print("--- Generated Answer ---")
print(response["answer"])

# Display Extracted PDF File & Page Citations
print("\n--- PDF Citations ---")
for i, doc in enumerate(response["context"], 1):
    source_path = doc.metadata.get("source", "Unknown PDF")
    pdf_filename = os.path.basename(source_path)
    page_number = doc.metadata.get("page", 0) + 1  # 1-indexed page display
    
    print(f"[{i}] Document: {pdf_filename} | Page: {page_number}")

Query: What is the return window and restocking fee for opened laptops?

--- Generated Answer ---
Based on the provided documentation, opened laptops have a **14-calendar-day return window** from the timestamp of delivery or store pickup. They are subject to a standard **15% restocking fee** if the protective factory film has been removed or if the device has been fully booted and configured.

--- PDF Citations ---
[1] Document: DOC_01_Return_and_Refund_Policy.pdf | Page: 1
[2] Document: DOC_01_Return_and_Refund_Policy.pdf | Page: 1
[3] Document: DOC_01_Return_and_Refund_Policy.pdf | Page: 2
